In [ ]:
class EaterEnvironment:
    def __init__(self, mover_policy, grid_size=7, g1_state=37, g2_state=49, initial_bananas_g1=100000, initial_bananas_g2=100000):
              self.mover_policy = mover_policy
              self.grid_size = grid_size
              self.g1 = self.state_to_position(g1_state)
              self.g2 = self.state_to_position(g2_state)
              self.bananas_g1 = initial_bananas_g1
              self.bananas_g2 = initial_bananas_g2
              self.actions = [
                  (1, 0), (0, 1), (0.5, 0.5), (0.2, 0.8), (0.6, 0.4),
                  (0.8, 0.2), (0.4, 0.6), (0.7, 0.3), (0.3, 0.7)
              ]
              # Initial position of the mover (can be set to a default or random for more generality)
              self.initial_mover_state = 1  # Example: starting at the bottom-left corner of the grid
              self.current_mover_state = self.initial_mover_state
              self.current_mover_action = 0  # Initial action can be arbitrarily set
              self.done = False

    def reset(self):
              # Reset the banana counts and the mover's position to the initial settings
              self.bananas_g1 = 100000
              self.bananas_g2 = 100000
              self.current_mover_state = 1
              self.current_mover_action = random.choice(range(0, 5))
              self.done  = False
              return (self.current_mover_state, self.current_mover_action)


    def state_to_position(self, state):
        x = (state - 1) % self.grid_size
        y = (state - 1) // self.grid_size
        return (x, y)

    def calculate_distance(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    def calculate_reward(self, distance_to_g1, distance_to_g2, action_index):
      if distance_to_g1 + distance_to_g2 == 0:
          return 0  # Avoid division by zero, although unlikely in practical scenarios

      # Ratio of how much should ideally be consumed from g1 based on its relative closeness
      ideal_g1_ratio = distance_to_g2 / (distance_to_g1 + distance_to_g2)
      ideal_g2_ratio = 1 - ideal_g1_ratio  # Complement of g1 ratio for g2

      # Actions defined with their consumption ratios from g1 and g2
      action_ratios = {
          0: (1.0, 0.0),  # All from g1
          1: (0.0, 1.0),  # All from g2
          2: (0.5, 0.5),  # Equally from both
          3: (0.2, 0.8),  # Mainly from g2
          4: (0.6, 0.4),  # Mainly from g1
          5: (0.8, 0.2),  # Mainly from g1
          6: (0.4, 0.6),  # Mainly from g2
          7: (0.7, 0.3),  # Mainly from g1
          8: (0.3, 0.7)   # Mainly from g2
      }

      actual_g1_ratio, actual_g2_ratio = action_ratios[action_index]

      # Calculate the absolute differences between actual ratios and ideal ratios
      diff_g1 = abs(actual_g1_ratio - ideal_g1_ratio)
      diff_g2 = abs(actual_g2_ratio - ideal_g2_ratio)

      # Evaluate the reward based on how well the actual ratios align with the ideal ratios
      reward = 1 - (diff_g1 + diff_g2)  # Reward is higher when the differences are smaller

      # Scale the reward to fit into specific reward levels you mentioned
      if reward > 0.95:
          return 1
      elif reward > 0.9:
          return 0.8
      elif reward > 0.85:
          return 0.7
      elif reward > 0.8:
          return 0.6
      elif reward > 0.75:
          return 0.4
      elif reward > 0.7:
          return 0.3
      elif reward > 0.65:
          return 0.2
      elif reward > 0.6:
          return 0
      else:
          return -100

    def step(self, eater_state, action_index):
            mover_current_state, _ = eater_state
            mover_position = self.state_to_position(mover_current_state)
            mover_next_action = self.mover_policy[mover_position]
            mover_next_position = self.update_position(mover_position, mover_next_action)
            next_mover_state = self.position_to_state(mover_next_position)
            next_mover_action = self.mover_policy[mover_next_position]

            distance_to_g1 = self.calculate_distance(mover_next_position, self.g1)
            distance_to_g2 = self.calculate_distance(mover_next_position, self.g2)
            goal1_consume, goal2_consume = self.actions[action_index]
            self.bananas_g1 -=   goal1_consume
            self.bananas_g2 -= goal2_consume
            reward = self.calculate_reward( distance_to_g1, distance_to_g2, action_index)
            next_state = (next_mover_state, next_mover_action)
            if  next_mover_action == 4:
              self.done = True
            return next_state, reward,self.done

    def update_position(self, position, action):
        x, y = position
        if action == 0 and y < self.grid_size - 1:
            y += 1
        elif action == 1 and y > 0:
            y -= 1
        elif action == 2 and x > 0:
            x -= 1
        elif action == 3 and x < self.grid_size - 1:
            x += 1
        return (x, y)

    def position_to_state(self, position):
        x, y = position
        return y * self.grid_size + x + 1


In [ ]:
mover_policy  = {
    (0, 0): 3, (0, 1): 3, (0, 2): 3, (0, 3): 3, (0, 4): 1, (0, 5): 3, (0, 6): 3,
    (1, 0): 3, (1, 1): 3, (1, 2): 3, (1, 3): 3, (1, 4): 3, (1, 5): 3, (1, 6): 3,
    (2, 0): 3, (2, 1): 3, (2, 2): 3, (2, 3): 3, (2, 4): 3, (2, 5): 3, (2, 6): 3,
    (3, 0): 3, (3, 1): 3, (3, 2): 3, (3, 3): 3, (3, 4): 3, (3, 5): 3, (3, 6): 1,
    (4, 0): 3, (4, 1): 3, (4, 2): 3, (4, 3): 3, (4, 4): 1, (4, 5): 1, (4, 6): 1,
    (5, 0): 3, (5, 1): 3, (5, 2): 3, (5, 3): 1, (5, 4): 1, (5, 5): 1, (5, 6): 1,
    (6, 0): 3, (6, 1): 3, (6, 2): 3, (6, 3): 1, (6, 4): 1, (6, 5): 1, (6,6):4
}


def state_to_position( state):
    x = (state - 1) % 7
    y = (state - 1) // 7
    return (x, y)
print(state_to_position(43))
mover_policy[state_to_position(44)]


(0, 6)


3

In [ ]:
import random
env = EaterEnvironment(mover_policy)
next_state = (1,0)
for i in range(10):
  action = random.choice(range(len(env.actions)))
  next_state, reward, done = env.step(next_state, action)
  if done == True:
    break




In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# Define the neural network model
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 16)
        self.fc2 = nn.Linear(16, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)



class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Assuming QNetwork and ReplayBuffer are defined above

state_dim = 2  # Define the size of the input state dimensions
action_dim = len(env.actions)  # Define the number of actions available in the environment
buffer_capacity = 10000
batch_size = 10
model = QNetwork(state_dim, action_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)
replay_buffer = ReplayBuffer(buffer_capacity)
gamma = 0.99
epsilon = 0.2
total_episodes = 5000

def compute_loss(batch):
    states, actions, rewards, next_states, dones = zip(*batch)
    states = torch.tensor(states, dtype=torch.float32)
    next_states = torch.tensor(next_states, dtype=torch.float32)
    actions = torch.tensor(actions)
    rewards = torch.tensor(rewards)
    dones = torch.tensor(dones)

    current_q_values = model(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    next_q_values = model(next_states).max(1)[0]
    next_q_values[dones] = 0.0  # Zero out the values for terminal states
    target_q_values = rewards + gamma * next_q_values

    loss = nn.MSELoss()(current_q_values, target_q_values)
    return loss

for epoch in range(total_episodes):
    state = env.reset()
    done = False
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = model(state_tensor)
        action = torch.argmax(q_values).item() if random.random() > epsilon else random.randint(0, action_dim - 1)
        next_state, reward, done = env.step(state, action)

        replay_buffer.push(state, action, reward, next_state, done)
        state = next_state

        # Regular update if enough samples are collected
        if len(replay_buffer) >= batch_size:
            batch = replay_buffer.sample(batch_size)
            loss = compute_loss(batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Update at the end of the episode using all available samples if fewer than batch_size
    if done and len(replay_buffer) < batch_size:
        batch = replay_buffer.sample(len(replay_buffer))  # Take all that is available
        loss = compute_loss(batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 100 == 0:
        print("Epoch:", epoch + 1, "Loss:", loss.item() if 'loss' in locals() else 'No update')
print("Final loss:", loss.item() if 'loss' in locals() else 'No final update')


Epoch: 1 Loss: 5498.9208984375
Epoch: 101 Loss: 634.7413330078125
Epoch: 201 Loss: 272.29620361328125
Epoch: 301 Loss: 78.07706451416016
Epoch: 401 Loss: 19.457788467407227
Epoch: 501 Loss: 13.548347473144531
Epoch: 601 Loss: 13.843584060668945
Epoch: 701 Loss: 15.244010925292969
Epoch: 801 Loss: 2.7594497203826904
Epoch: 901 Loss: 179.9538116455078
Epoch: 1001 Loss: 438.66583251953125
Epoch: 1101 Loss: 21.807973861694336
Epoch: 1201 Loss: 10.958288192749023
Epoch: 1301 Loss: 25.68804359436035
Epoch: 1401 Loss: 311.03436279296875
Epoch: 1501 Loss: 14.62980842590332
Epoch: 1601 Loss: 31.64593505859375
Epoch: 1701 Loss: 25.670536041259766
Epoch: 1801 Loss: 1.966239333152771
Epoch: 1901 Loss: 71.87120056152344
Epoch: 2001 Loss: 162.5325164794922
Epoch: 2101 Loss: 32.14332580566406
Epoch: 2201 Loss: 91.41035461425781
Epoch: 2301 Loss: 7.339764595031738
Epoch: 2401 Loss: 284.16717529296875
Epoch: 2501 Loss: 11.033536911010742
Epoch: 2601 Loss: 3.2097525596618652
Epoch: 2701 Loss: 3.88283801

In [ ]:
policy = {}
for mover_position in range(1, 50):
  for mover_action in range(4):
    state = (mover_position, mover_action)
    state_tensor = torch.tensor(state, dtype=torch.float32)
    q_values = model(state_tensor)
    best_action = torch.argmax(q_values).item()
    policy[state] = best_action
print(policy)

{(1, 0): 7, (1, 1): 7, (1, 2): 7, (1, 3): 7, (2, 0): 7, (2, 1): 7, (2, 2): 7, (2, 3): 7, (3, 0): 7, (3, 1): 7, (3, 2): 7, (3, 3): 7, (4, 0): 7, (4, 1): 7, (4, 2): 7, (4, 3): 7, (5, 0): 7, (5, 1): 7, (5, 2): 7, (5, 3): 7, (6, 0): 7, (6, 1): 7, (6, 2): 7, (6, 3): 7, (7, 0): 7, (7, 1): 7, (7, 2): 7, (7, 3): 7, (8, 0): 7, (8, 1): 7, (8, 2): 7, (8, 3): 7, (9, 0): 7, (9, 1): 7, (9, 2): 7, (9, 3): 7, (10, 0): 7, (10, 1): 7, (10, 2): 7, (10, 3): 7, (11, 0): 7, (11, 1): 7, (11, 2): 7, (11, 3): 7, (12, 0): 7, (12, 1): 7, (12, 2): 7, (12, 3): 7, (13, 0): 7, (13, 1): 7, (13, 2): 7, (13, 3): 7, (14, 0): 7, (14, 1): 7, (14, 2): 7, (14, 3): 7, (15, 0): 7, (15, 1): 7, (15, 2): 7, (15, 3): 7, (16, 0): 5, (16, 1): 7, (16, 2): 7, (16, 3): 7, (17, 0): 7, (17, 1): 7, (17, 2): 7, (17, 3): 5, (18, 0): 7, (18, 1): 7, (18, 2): 7, (18, 3): 5, (19, 0): 7, (19, 1): 7, (19, 2): 5, (19, 3): 5, (20, 0): 7, (20, 1): 7, (20, 2): 5, (20, 3): 5, (21, 0): 7, (21, 1): 5, (21, 2): 5, (21, 3): 7, (22, 0): 7, (22, 1): 7, (22

In [ ]:
env = EaterEnvironment(mover_policy)
state = env.reset()

for i in range(19):
    if state[1] == 4:
      print(f" Final goal : {state[0]} Bananas at g1: {env.bananas_g1}, Bananas at g2: {env.bananas_g2}")
      break
    action_index = policy[state] # Cycle through actions
    next_state, reward, done = env.step(state, action_index)
    if done != True:
      print(f"Step {i+1}: Next State: {next_state}, Reward: {reward}, action: {env.actions[action_index]} Bananas at g1: {env.bananas_g1}, Bananas at g2: {env.bananas_g2}")
    state = next_state

 Final goal : 1 Bananas at g1: 100000, Bananas at g2: 100000
